# 4 - Deployment and testing in the cloud

Notebooks 1-3 showed how the data is collected and prepared and how the models are
trained and evaluated. This notebook covers the other half of the task: how the
solution is **deployed on Google Cloud** and how it is **tested**, before and after
every release.

1. Deployment architecture and release flow
2. Testing plan
3. Pre-release tests (run here: the automated test suite and lint)
4. Post-deployment tests against the live service
   [climatelens.sulemanb.com](https://climatelens.sulemanb.com)

In [1]:
import os
import statistics
import subprocess
import sys
import time
from datetime import timedelta
from pathlib import Path

import httpx
import pandas as pd

from climatelens import __version__
from climatelens.predict import local_today

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
LIVE = "https://climatelens.sulemanb.com"
CLOUD_RUN = "https://climatelens-273124305778.europe-west3.run.app"
print("code version in this checkout:", __version__)

code version in this checkout: 1.3.1


## 4.1 Deployment architecture

![Cloud architecture](../diagrams/architecture.svg)

*(Diagram: [`diagrams/architecture.svg`](../diagrams/architecture.svg).)*

| Path | Flow |
|---|---|
| Request | visitor -> Cloudflare DNS -> Firebase Hosting (HTTPS, CDN cache 5 min) -> Cloud Run service `climatelens` (FastAPI, scales to zero) |
| Batch | Cloud Scheduler -> `POST /jobs/ingest` daily 15:30 and `POST /jobs/train` monthly, authenticated with Google-signed OIDC tokens -> Cloud Storage |
| Delivery | protected git tag `vX.Y.Z` -> GitLab CI -> Workload Identity Federation (no keys) -> Artifact Registry (immutable image) -> Cloud Run (deploy by digest) |

All 41 cloud resources are defined in Terraform (`infra/`); nothing is created by
hand.

### Release flow

| Stage | Job | What it does |
|---|---|---|
| lint | `lint` | `ruff` code style and static checks |
| test | `test` | the automated test suite with coverage |
| release | `release-image` | checks that the tag equals the code version, builds the image, smoke-tests it, pushes `vX.Y.Z` to Artifact Registry |
| deploy | `deploy` | deploys exactly that image by digest to Cloud Run and checks the live service |

A release is one command once the version is bumped and committed:
`make release` (creates and pushes the tag). Branch pipelines run lint, test and a
container build with smoke test for every code change.

## 4.2 Testing plan

| Level | Where | What it proves |
|---|---|---|
| Static checks | CI `lint` | consistent, error-free code (`ruff`) |
| Unit and integration tests | CI `test`, locally `make test` | data validation, features (incl. no look-ahead), models, evaluation, pipeline, storage (in-memory fake of Cloud Storage), API and job authentication |
| Container smoke test | CI `container` / `release-image` | image runs as non-root, LightGBM loads, `/health` answers, service fails closed (503) without a model |
| Release checks | CI `release-image` | git tag equals code version; registry rejects re-used tags |
| Post-deployment checks | CI `deploy`, and section 4.4 below | the live service answers with the new version |
| Model quality gate | monthly train job | a retrained model is published only if no forecast is more than 5 % worse |
| Operations | Cloud Logging, budget alert | job results logged as JSON; alert at EUR 10/month |

## 4.3 Pre-release tests

The same commands the CI `lint` and `test` jobs run.

In [2]:
# Plain output (no terminal colours) so it reads well in the saved notebook.
PLAIN = {k: v for k, v in os.environ.items() if k != "FORCE_COLOR"} | {"NO_COLOR": "1"}


def run(*args: str) -> str:
    result = subprocess.run(
        [sys.executable, "-m", *args], capture_output=True, text=True, env=PLAIN
    )
    return f"exit code {result.returncode}\n" + (result.stdout + result.stderr).strip()


print(run("ruff", "check", "."))

exit code 0
All checks passed!


In [3]:
start = time.perf_counter()
output = run("pytest", "-q", "--color=no", "-p", "no:cacheprovider")
print("\n".join(output.splitlines()[-4:]))
print(f"({time.perf_counter() - start:.0f} s)")

    parser.resetCache()

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
99 passed, 14 warnings in 38.34s
(40 s)


## 4.4 Post-deployment tests against the live service

Each check calls the public service exactly as a visitor or Cloud Scheduler would.

In [4]:
client = httpx.Client(timeout=30.0)
tomorrow = (local_today() + timedelta(days=1)).isoformat()
checks = []


def check(name: str, ok: bool, detail: object = "") -> None:
    checks.append({"check": name, "result": "pass" if ok else "FAIL", "detail": detail})


r = client.get(f"{LIVE}/health")
check("health endpoint answers", r.status_code == 200 and r.json() == {"status": "ok"}, r.text)
check(
    "health is never cached",
    r.headers.get("cache-control") == "no-store",
    r.headers.get("cache-control"),
)
check(
    "HTTPS enforced with HSTS",
    "strict-transport-security" in r.headers,
    r.headers.get("strict-transport-security"),
)

r = client.get(f"{LIVE}/?city=berlin&city=munich&city=hamburg")
check(
    "dashboard renders",
    r.status_code == 200 and "ClimateLens" in r.text,
    f"{r.status_code}, {len(r.text):,} bytes",
)
check(
    "dashboard cached by the CDN",
    "public" in r.headers.get("cache-control", ""),
    r.headers.get("cache-control"),
)

r = client.get(f"{LIVE}/cities")
check("12 cities configured", r.status_code == 200 and len(r.json()) == 12, len(r.json()))

forecast = client.get(f"{LIVE}/forecast").json()
items = forecast["forecasts"]
items = list(items.values()) if isinstance(items, dict) else items
check("forecast for every city", len(items) == 12, len(items))
check(
    "forecasts are for tomorrow",
    all(f["valid_for"] == tomorrow for f in items),
    f"valid_for {sorted({f['valid_for'] for f in items})}, tomorrow {tomorrow}",
)

r = client.get(f"{LIVE}/forecast/atlantis")
check("unknown city rejected", r.status_code == 404, r.status_code)

metrics = client.get(f"{LIVE}/metrics").json()
check(
    "metrics match the served model",
    metrics["model_version"] == forecast["model_version"],
    metrics["model_version"],
)

r = client.post(f"{LIVE}/jobs/ingest")
check("ingest job refuses anonymous calls", r.status_code == 401, r.status_code)
r = client.post(f"{LIVE}/jobs/train", headers={"Authorization": "Bearer not-a-real-token"})
check("train job refuses a forged token", r.status_code == 401, r.status_code)

r = client.get(f"{CLOUD_RUN}/health")
check("Cloud Run service URL answers", r.status_code == 200, r.status_code)

live_checks = pd.DataFrame(checks)
print(f"{(live_checks['result'] == 'pass').sum()} of {len(live_checks)} live checks passed")
live_checks

13 of 13 live checks passed


,check,result,detail
0,health endpoint answers,pass,"{""status"":""ok""}"
1,health is never cached,pass,no-store
2,HTTPS enforced with HSTS,pass,max-age=31556926
3,dashboard renders,pass,"200, 27,527 bytes"
4,dashboard cached by the CDN,pass,"public, max-age=60, s-maxage=300"
5,12 cities configured,pass,12
6,forecast for every city,pass,12
7,forecasts are for tomorrow,pass,"valid_for ['2026-10-19'], tomorrow 2026-10-19"
8,unknown city rejected,pass,404
9,metrics match the served model,pass,20261017T131809Z


### Response times

Repeated requests from this machine. Forecasts are precomputed into a small
snapshot after every update, so a request only reads a file; repeat visits are
served from the CDN.

In [5]:
def timings(url: str, n: int = 10) -> list[float]:
    out = []
    for _ in range(n):
        t0 = time.perf_counter()
        client.get(url).raise_for_status()
        out.append(time.perf_counter() - t0)
    return out


latency = pd.DataFrame(
    {
        path: {"median s": statistics.median(t), "max s": max(t)}
        for path in ("/health", "/forecast", "/?city=berlin")
        for t in [timings(LIVE + path)]
    }
).T.round(3)
latency

,median s,max s
/health,0.082,0.087
/forecast,0.033,0.040
/?city=berlin,0.045,0.107


### Today's live forecasts

In [6]:
pd.DataFrame(
    [
        {
            "city": f["name"],
            "based on data of": f["based_on"],
            "valid for": f["valid_for"],
            "days ahead": f["horizon_days"],
            "frost chance": f"{f['frost']['probability']:.0%}",
            "frost risk": f["frost"]["risk"],
            "sunshine kWh/m²": f["solar"]["kwh_m2"],
            "vs usual": f["solar"]["outlook"],
            "wind km/h": f["wind"]["speed_kmh"],
            "Beaufort": f"{f['wind']['beaufort']} - {f['wind']['description']}",
        }
        for f in items
    ]
)

,city,based on data of,valid for,days ahead,frost chance,frost risk,sunshine kWh/m²,vs usual,wind km/h,Beaufort
0,Berlin,2026-10-16,2026-10-19,3,6%,low,1.64,typical,12,3 - Gentle breeze
1,Hamburg,2026-10-16,2026-10-19,3,2%,low,1.50,typical,14,3 - Gentle breeze
2,Munich,2026-10-16,2026-10-19,3,0%,low,1.70,typical,8,2 - Light breeze
3,Cologne,2026-10-16,2026-10-19,3,2%,low,1.68,typical,12,3 - Gentle breeze
4,Frankfurt am Main,2026-10-16,2026-10-19,3,2%,low,1.66,typical,7,2 - Light breeze
5,Stuttgart,2026-10-16,2026-10-19,3,0%,low,1.80,typical,9,2 - Light breeze
6,Essen,2026-10-16,2026-10-19,3,0%,low,1.58,typical,11,2 - Light breeze
7,Leipzig,2026-10-16,2026-10-19,3,6%,low,1.66,typical,8,2 - Light breeze
8,Bremen,2026-10-16,2026-10-19,3,0%,low,1.50,typical,14,3 - Gentle breeze
9,Dresden,2026-10-16,2026-10-19,3,2%,low,1.65,typical,16,3 - Gentle breeze


### Accuracy of the served models (from the live `/metrics` endpoint)

The deployed service publishes the walk-forward scores of the bundle it serves -
the same evaluation as notebook 3, run in the cloud by the monthly training job.

In [7]:
print("served model bundle:", metrics["model_version"])
pd.DataFrame(
    [
        {
            "forecast": m["target"].removeprefix("target_"),
            "days ahead": m["horizon_days"],
            "served model": m["served_model"],
            "metric": m["primary_metric"],
            "score": m["score"],
            "reference": m["reference_baseline"],
            "reference score": m["reference_score"],
            "skill": f"{m['skill']:+.1%}",
            "test period": f"{m['test_start']} .. {m['test_end']}",
        }
        for m in metrics["targets"].values()
    ]
)

served model bundle: 20261017T131809Z


,forecast,days ahead,served model,metric,score,reference,reference score,skill,test period
0,frost,1,lightgbm,brier,0.0564,climatology,0.1147,+50.8%,2024-10-01 .. 2026-10-15
1,frost,2,lightgbm,brier,0.1018,climatology,0.1144,+11.0%,2024-10-01 .. 2026-10-14
2,frost,3,climatology,brier,0.1144,climatology,0.1144,+0.0%,2024-10-01 .. 2026-10-13
3,solar_kwh_m2,1,lightgbm,mae,0.8427,persistence,0.9419,+10.5%,2024-10-01 .. 2026-07-30
4,solar_kwh_m2,2,lightgbm,mae,0.9634,persistence,1.1153,+13.6%,2024-10-01 .. 2026-07-29
5,solar_kwh_m2,3,lightgbm,mae,1.0224,persistence,1.2035,+15.0%,2024-10-01 .. 2026-07-28
6,wind_speed,1,lightgbm,mae,0.8290,persistence,0.9201,+9.9%,2024-10-01 .. 2026-10-15
7,wind_speed,2,lightgbm,mae,0.9650,persistence,1.1161,+13.5%,2024-10-01 .. 2026-10-14
8,wind_speed,3,climatology,mae,0.9961,persistence,1.1734,+15.1%,2024-10-01 .. 2026-10-13


## Summary

| Requirement | Evidence in this notebook |
|---|---|
| Deployed in the cloud | live service on Cloud Run behind Firebase Hosting, custom domain with HTTPS |
| Automated, repeatable deployment | Terraform for all resources; GitLab CI releases on protected tags, keyless |
| Tested before release | lint and the full automated test suite (section 4.3) |
| Tested after release | live checks: health, dashboard, API, tomorrow's date, caching, job protection (section 4.4) |
| Model quality in production | live `/metrics`; quality gate on every monthly retrain |